In [ ]:
import numpy as np
import pandas as pd
import librosa
import os
from pathlib import Path
from tqdm import tqdm
from scipy.stats import skew, kurtosis

In [ ]:
# Production copy note:
# The final CSV dataset is frozen for publication. Re-run this notebook only if you intentionally rebuild the dataset.
# Paths can be overridden with environment variables so the notebook is not tied to one machine.

ai_dir = "datasets/ai_TOUSE_music_dataset"
human_dir = "datasets/human_TOUSE_music_dataset"

OUTPUT_CSV = 'FINAL_ai_human_music_detector_dataset.csv'

target_sr = 22050
window_duration = 30  # seconds


In [9]:
def preprocess_audio(filepath) : 
    # Loads audio file, converts to mono, resamples, and extracts center 20-second window.

    # To load the audio file
    y, sr = librosa.load(filepath, sr = target_sr, mono = True)
    total_duration = len(y) / sr

    # Take the center window duration
    if total_duration <= window_duration :
        return y, sr
        
    window_samples = window_duration * sr
    center = len(y) // 2
    
    start = center - window_samples // 2
    end = start + window_samples
    y_window = y[start:end]
    
    return y_window, sr

In [10]:
def extract_features(filepath):

    # Preprocess
    y, sr = preprocess_audio(filepath)

    # RMS Energy
    rms = librosa.feature.rms(y=y).flatten()

    # Spectral Features
    centroid  = librosa.feature.spectral_centroid(y=y, sr=sr).flatten()
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr).flatten()
    rolloff   = librosa.feature.spectral_rolloff(y=y, sr=sr).flatten()

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y).flatten()

    # Chroma — 12 pitch classes
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)

    # Tempo and Beat Count
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr)

    # MFCCs & NEW: Delta MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_delta = librosa.feature.delta(mfcc)

    # Dynamic Range
    dynamic_range   = float(np.max(rms) - np.min(rms))
    rms_variability = float(np.std(rms) / (np.mean(rms) + 1e-9))

    # Spectral Flatness
    flatness = librosa.feature.spectral_flatness(y=y).flatten()

    # Onset Strength
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)

    # Tonnetz 
    y_harmonic = librosa.effects.harmonic(y)
    tonnetz = librosa.feature.tonnetz(y=y_harmonic, sr=sr)

    # Build Feature Vector
    features = {
        # RMS Features
        "rms_mean":       np.mean(rms),
        "rms_var":        np.var(rms),
        "rms_skew":       skew(rms),         
        "rms_kurtosis":   kurtosis(rms),     

        # Spectral Centroid Features
        "centroid_mean":  np.mean(centroid),
        "centroid_var":   np.var(centroid),
        "centroid_skew":  skew(centroid),     
        "centroid_kurtosis": kurtosis(centroid), 

        "bandwidth_mean": np.mean(bandwidth),
        "bandwidth_var":  np.var(bandwidth),

        "rolloff_mean":   np.mean(rolloff),
        "rolloff_var":    np.var(rolloff),

        "zcr_mean":       np.mean(zcr),
        "zcr_var":        np.var(zcr),

        "tempo":          float(tempo[0]),
        "num_beats":      len(beats),

        "dynamic_range":   dynamic_range,
        "rms_variability": rms_variability,

        # Spectral Flatness Features
        "flatness_mean":  np.mean(flatness),
        "flatness_var":   np.var(flatness),
        "flatness_skew":  skew(flatness),      
        "flatness_kurtosis": kurtosis(flatness), 

        # Onset Strength Features
        "onset_mean":     np.mean(onset_env),
        "onset_var":      np.var(onset_env),
        "onset_skew":     skew(onset_env),     
        "onset_kurtosis": kurtosis(onset_env), 

        "tonnetz_mean":   np.mean(tonnetz),
        "tonnetz_var":    np.var(tonnetz),
    }

    # Chroma features (24 features)
    for i in range(12):
        features[f"chroma_{i+1}_mean"] = np.mean(chroma[i])
        features[f"chroma_{i+1}_var"]  = np.var(chroma[i])

    # MFCC & Delta MFCC features (52 features)
    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_var"]  = np.var(mfcc[i])
        
        # Delta MFCC tracking
        features[f"mfcc_delta_{i+1}_mean"] = np.mean(mfcc_delta[i])
        features[f"mfcc_delta_{i+1}_var"]  = np.var(mfcc_delta[i])

    return features

In [11]:
data = []

# Processing Human Songs
for file in tqdm(os.listdir(human_dir)):
    if file.endswith((".wav", ".mp3")):
        filepath = os.path.join(human_dir, file)
        try:
            features = extract_features(filepath)
            features["label"] = 0
            features["filename"] = file        # ADD THIS
            data.append(features)
        except Exception as e:
            print(f"Skipping {file} due to error: {e}")

# Processing AI Songs
for file in tqdm(os.listdir(ai_dir)):
    if file.endswith((".wav", ".mp3")):
        filepath = os.path.join(ai_dir, file)
        try:
            features = extract_features(filepath)
            features["label"] = 1
            features["filename"] = file        # ADD THIS
            data.append(features)
        except Exception as e:
            print(f"Skipping {file} due to error: {e}")

df = pd.DataFrame(data)
print("Dataset shape:", df.shape)
print(df["label"].value_counts().rename({0: "Human", 1: "AI"}))
df.head()

100%|████████████████████████████████████████████████████████████████████████████████| 220/220 [15:51<00:00,  4.32s/it]

Dataset shape: (440, 106)
label
Human    220
AI       220
Name: count, dtype: int64


,rms_mean,rms_var,rms_skew,rms_kurtosis,centroid_mean,centroid_var,centroid_skew,centroid_kurtosis,bandwidth_mean,bandwidth_var,...,mfcc_12_mean,mfcc_12_var,mfcc_delta_12_mean,mfcc_delta_12_var,mfcc_13_mean,mfcc_13_var,mfcc_delta_13_mean,mfcc_delta_13_var,label,filename
0,0.142919,0.009173,1.263832,2.710377,1412.302453,205264.421182,0.651066,0.469132,1513.961835,145396.042954,...,1.788929,58.210636,0.007574,1.220569,-3.004418,93.329605,-0.003183,1.776388,0,blues_blues.00010.wav
1,0.113409,0.003689,1.029931,1.640340,1156.214859,314318.424802,1.125311,1.053324,1498.239871,59158.977340,...,-2.222806,77.469498,0.019466,1.573031,3.923156,65.911003,-0.003723,1.361513,0,blues_blues.00022.wav
2,0.095752,0.002192,0.687863,0.310933,1391.693584,200465.952492,0.584892,-0.519092,1494.974017,43595.784812,...,-4.506482,58.997662,0.001515,1.569855,-0.251702,67.889923,0.000838,1.771786,0,blues_blues.00027.wav
3,0.099567,0.004087,1.502175,3.031433,1115.755561,106163.992154,0.511980,-0.310419,1399.289200,53128.054270,...,-2.792608,67.013397,0.000156,1.964452,-0.903065,95.245323,0.001743,2.210567,0,blues_blues.00028.wav
4,0.170050,0.003191,0.554323,0.262315,1379.107222,552351.632806,2.335833,7.589688,2003.609479,269452.425510,...,-7.924602,60.308189,-0.006955,2.008784,-17.706299,73.099495,-0.009622,1.965097,0,blues_blues.00030.wav


In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f'Dataset saved: {OUTPUT_CSV}')